---
## 1. Setup & Imports

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
import time

# Import modular components
from src.config import RANDOM_STATE, PCA_FEATURES, COLORS
from src.data_loader import load_data, inspect_data, clean_data, print_class_distribution, compare_fraud_stats
from src.preprocessing import engineer_features, get_feature_columns, prepare_features, split_data, scale_features, apply_smote
from src.visualization import (plot_class_distribution, plot_amount_distribution, plot_time_distribution,
                               plot_correlation_heatmap, plot_violin_features, analyze_outliers,
                               plot_pca_variance, plot_pca_2d, plot_pca_3d, plot_feature_importance,
                               plot_smote_effect)
from src.clustering import kmeans_elbow_analysis, kmeans_clustering, isolation_forest_analysis, dbscan_analysis
from src.models import evaluate, train_classical_models, train_ensemble_models, tune_xgboost, tune_lightgbm
from src.deep_learning import train_mlp, train_autoencoder, train_lstm, train_cnn
from src.evaluation import (create_comparison_table, plot_confusion_matrices, plot_roc_curves,
                            plot_pr_curves, plot_model_comparison, plot_shap_analysis,
                            print_best_model_report, print_final_summary)

print('All libraries and modules imported successfully!')

In [ ]:
# Load the dataset
df = load_data()
df.head()

---
## 2. Data Cleaning & Preprocessing

In [ ]:
# Basic dataset info
inspect_data(df)

In [ ]:
# Clean data: check missing values, duplicates, and infinite values
df = clean_data(df)

In [ ]:
# Class distribution
print_class_distribution(df)

In [ ]:
# Compare fraud vs legitimate statistics
compare_fraud_stats(df)

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution plot
plot_class_distribution(df)

In [ ]:
# Transaction Amount distribution
plot_amount_distribution(df)

In [ ]:
# Time distribution
plot_time_distribution(df)

In [ ]:
# Correlation heatmap — returns target_corr for later use
target_corr = plot_correlation_heatmap(df)

In [ ]:
# Violin plots for top discriminative features
plot_violin_features(df, target_corr)

In [ ]:
# Outlier analysis using IQR method
analyze_outliers(df, target_corr)

---
## 4. PCA Analysis & Feature Importance

In [ ]:
# Variance analysis of PCA components
plot_pca_variance(df)

In [ ]:
# 2D PCA visualisation — returns X_2d for clustering
X_2d = plot_pca_2d(df)

In [ ]:
# 3D PCA visualisation (interactive)
plot_pca_3d(df)

In [ ]:
# Feature importance using Mann-Whitney U test
corr_matrix = df.corr()
plot_feature_importance(df, corr_matrix)

---
## 5. Feature Engineering & Scaling

In [ ]:
# Create new features
df = engineer_features(df)

In [ ]:
# Define features and target
X, y = prepare_features(df)

---
## 6. Train-Test Split

In [ ]:
# Split data (80% train, 20% test) with stratification
X_train, X_test, y_train, y_test = split_data(X, y)

In [ ]:
# StandardScaler - fit on training data ONLY to prevent data leakage
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

---
## 7. Handling Class Imbalance with SMOTE

In [ ]:
# Apply SMOTE on training data ONLY
X_train_smote, y_train_smote = apply_smote(X_train_scaled, y_train)
print(f'\nTest data unchanged: {len(X_test_scaled):,} samples')

In [ ]:
# Visualise the effect of SMOTE
plot_smote_effect(y_train, y_train_smote)

---
## 8. Clustering Analysis

In [ ]:
# K-Means: find optimal K
best_k = kmeans_elbow_analysis(X_2d)

In [ ]:
# K-Means clustering and fraud analysis
kmeans_clustering(X_2d, df, best_k)

In [ ]:
# Isolation Forest - unsupervised anomaly detection
isolation_forest_analysis(X_2d, df)

In [ ]:
# DBSCAN - density-based clustering
dbscan_analysis(X_2d, df)

---
## 9. Model Training & Comparison

In [ ]:
# Initialise result tracking
all_results = []
all_preds = {}
all_probs = {}

### 9A. Classical ML Models

In [ ]:
# Train classical ML models: Logistic Regression, Decision Tree, Random Forest, SVM, KNN
classical_models = train_classical_models(X_train_smote, y_train_smote, X_test_scaled, y_test,
                                          all_results, all_preds, all_probs)

### 9B. Advanced Ensemble Models

In [ ]:
# Train ensemble models: XGBoost and LightGBM
ensemble_models = train_ensemble_models(X_train_smote, y_train_smote, X_test_scaled, y_test,
                                        all_results, all_preds, all_probs)

### 9C. Deep Learning Models

In [ ]:
# Neural Network (MLP)
mlp = train_mlp(X_train_smote, y_train_smote, X_test_scaled, y_test,
                evaluate, all_results, all_preds, all_probs)

In [ ]:
# Autoencoder - trained on legitimate transactions only, detects fraud as anomalies
autoencoder = train_autoencoder(X_train_scaled, y_train, X_test_scaled, y_test,
                                evaluate, all_results, all_preds, all_probs)

In [ ]:
# LSTM - treats features as a sequence
lstm_model = train_lstm(X_train_smote, y_train_smote, X_test_scaled, y_test,
                        evaluate, all_results, all_preds, all_probs)

In [ ]:
# 1D-CNN
cnn_model = train_cnn(X_train_smote, y_train_smote, X_test_scaled, y_test,
                      evaluate, all_results, all_preds, all_probs)

### 9D. Hyperparameter Tuning (Optuna)

In [ ]:
# Tune XGBoost with Optuna
xgb_tuned = tune_xgboost(X_train_smote, y_train_smote, X_test_scaled, y_test,
                         all_results, all_preds, all_probs, n_trials=30)

In [ ]:
# Tune LightGBM with Optuna
lgbm_tuned = tune_lightgbm(X_train_smote, y_train_smote, X_test_scaled, y_test,
                           all_results, all_preds, all_probs, n_trials=30)

---
## 10. Evaluation & Results

In [ ]:
# Model comparison table
results_df = create_comparison_table(all_results)

In [ ]:
# Confusion matrices for all models
plot_confusion_matrices(all_preds, y_test)

In [ ]:
# ROC curves
plot_roc_curves(all_probs, y_test)

In [ ]:
# Precision-Recall curves
plot_pr_curves(all_probs, y_test)

In [ ]:
# Model comparison bar chart
plot_model_comparison(results_df)

In [ ]:
# SHAP explainability
plot_shap_analysis(xgb_tuned, X_test_scaled)

In [ ]:
# Classification report for best model
print_best_model_report(results_df, all_preds, y_test)

In [ ]:
# Final Summary
print_final_summary(results_df, df)